# Practical 9: Image Features and CBIR

**Aim:** Extract color and texture features, plot a histogram, and retrieve images similar to a query.

This notebook reads PNG images from `images/`, using `query.png` as the query. It includes four simple sample images. Install the libraries if needed: `python -m pip install opencv-python ImageHash Pillow matplotlib`. Run all cells from top to bottom with this practical folder as the working directory.


## 1. Extract features

The hue histogram represents color, grayscale standard deviation is a simple texture measure, and pHash represents image structure.


In [ ]:
from pathlib import Path

import cv2
import imagehash
import matplotlib.pyplot as plt
from PIL import Image

IMAGE_DIR = Path("images")
QUERY = IMAGE_DIR / "query.png"
TOP_K = 3

def features(path):
    image = cv2.imread(str(path))
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    color_hist = cv2.calcHist([hsv], [0], None, [16], [0, 180])
    color_hist = cv2.normalize(color_hist, color_hist).flatten()
    texture = gray.std()
    image_hash = imagehash.phash(Image.open(path))
    return color_hist, texture, image_hash

query_hist, query_texture, query_hash = features(QUERY)


## 2. Find similar images

The program adds normalized color, texture, and hash distances. Smaller combined distance means a closer match.


In [ ]:
results = []
for path in sorted(IMAGE_DIR.glob("*.png")):
    if path.name == QUERY.name:
        continue
    hist, texture, image_hash = features(path)
    color_distance = cv2.compareHist(query_hist, hist, cv2.HISTCMP_BHATTACHARYYA)
    texture_distance = abs(query_texture - texture) / 255
    hash_distance = (query_hash - image_hash) / 64
    results.append((color_distance + texture_distance + hash_distance, path))

results.sort(key=lambda result: result[0])
for rank, (distance, path) in enumerate(results[:TOP_K], start=1):
    print(rank, path.name, round(distance, 3))


## 3. Plot features and show the results


In [ ]:
plt.plot(query_hist)
plt.title("Query image color histogram")
plt.xlabel("Hue bin")
plt.ylabel("Frequency")
plt.show()

fig, axes = plt.subplots(1, TOP_K + 1, figsize=(12, 3))
for axis, (title, path) in zip(
        axes, [("Query", QUERY)] + [(p.name, p) for score, p in results[:TOP_K]]):
    image = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
plt.show()


## Related applications

- **Spotify-style music discovery:** compare audio features to find tracks with similar sound. This uses the retrieval idea, but this notebook's features are for images.
- **YouTube-style video retrieval:** compare visual features from frames or thumbnails to find visually similar content.


## Viva questions with answers

1. **What is CBIR?** Content-Based Image Retrieval finds images by comparing their visual features.
2. **What is a color histogram?** A count of how many pixels fall into each color range.
3. **What does HSV stand for?** Hue, Saturation, and Value.
4. **What texture feature is extracted here?** The standard deviation of grayscale pixel values, used as a simple measure of image variation.
5. **What is a perceptual hash?** A compact fingerprint designed to represent an image's visual structure.
6. **What does hash distance tell us?** The number of differing hash bits; smaller distance usually means more similar images.
7. **Why sort distances from smallest to largest?** Smaller feature distance indicates a closer match to the query.
8. **What does the histogram plot show?** The distribution of hue values in the query image.
9. **Name one video retrieval use case.** Finding visually similar video frames or thumbnails.
10. **How can music search use a similar idea?** It can compare audio features to find tracks with similar sound, using audio rather than image features.
